In [1]:
import sys
sys.path.append("../IMAGE-Mat/scripts/vema")
from preprocessing import preprocessing


In [2]:
import prism

In [3]:
base_dir = "../IMAGE-Mat/scripts/vema"
preprocessing_results = preprocessing(base_dir)

/home/qubix/Documents/repos/image-mat/image-materials/prism-stock-classes/../IMAGE-Mat/scripts/vema/preprocessing.py:311: PerformanceWarning: indexing past lexsort depth may impact performance.
  vehicle_shares_typical[('Cars', fuel)] = \
/home/qubix/Documents/repos/image-mat/image-materials/prism-stock-classes/../IMAGE-Mat/scripts/vema/preprocessing.py:311: PerformanceWarning: indexing past lexsort depth may impact performance.
  vehicle_shares_typical[('Cars', fuel)] = \
/home/qubix/Documents/repos/image-mat/image-materials/prism-stock-classes/../IMAGE-Mat/scripts/vema/preprocessing.py:311: PerformanceWarning: indexing past lexsort depth may impact performance.
  vehicle_shares_typical[('Cars', fuel)] = \
/home/qubix/Documents/repos/image-mat/image-materials/prism-stock-classes/../IMAGE-Mat/scripts/vema/preprocessing.py:311: PerformanceWarning: indexing past lexsort depth may impact performance.
  vehicle_shares_typical[('Cars', fuel)] = \
/home/qubix/Documents/repos/image-mat/image-

In [4]:
vehicle_nr = preprocessing_results['total_nr_vehicles_simple']
life_time_vehicles = preprocessing_results["lifetimes_vehicles"].to_xarray()

In [5]:
from survival import FoldedNormalSurvival, SurvivalMatrix

survival = FoldedNormalSurvival(life_time_vehicles)
survival_matrix = SurvivalMatrix(survival)


In [6]:
from globals.constants import _IMAGE_REGIONS
import numpy as np

In [7]:
import xarray as xr

In [8]:
def convert_vehicles(vehicle_nr):
        xr_vehicles_orig = vehicle_nr.to_xarray()
        time_series = xr_vehicles_orig.coords["time"]
        modes = np.unique([x[0] for x in xr_vehicles_orig.data_vars])
        region_dim = np.unique([x[1] for x in xr_vehicles_orig.data_vars])


        xr_vehicles = xr.DataArray(0.0, dims=("time", "mode", "region"),
                coords={"time": time_series,
                        "mode": modes,
                        "region": region_dim})
        for dv in xr_vehicles_orig.data_vars:
                xr_vehicles.loc[:, dv[0], dv[1]] = xr_vehicles_orig.data_vars[dv]
        return xr_vehicles


In [9]:
xr_vehicle_nr = convert_vehicles(vehicle_nr)

In [10]:
def compute_historic(input_stock, survival, start_simulation, stock_by_cohort, inflow, outflow_by_cohort):
    first_year = xr_vehicle_nr.coords["time"][0]
    for t in xr_vehicle_nr.coords["time"].loc[first_year+1:1970]:
        stock_diff = input_stock.loc[t] - stock_by_cohort[t].sum("cohort")
        inflow[t] = stock_diff
        stock_by_cohort[t] = inflow[t]*survival[t, :]
        outflow_by_cohort[t] = stock_by_cohort[t]-stock_by_cohort[t-1]

In [11]:
Region = prism.NewDim("region", coords=[str(x) for x in np.arange(1, 27)])
Mode = prism.NewDim("mode", coords=list(vehicle_nr.columns.levels[0].unique()))
Cohort = prism.NewDim("cohort", coords=[str(x) for x in vehicle_nr.index.to_numpy()])

In [12]:
@prism.interface
class Stocks(prism.Model):
    start_simulation: int
    survival_matrix: SurvivalMatrix
    input_stock: prism.TimeVariable[Region, Mode]

    stock_by_cohort: prism.TimeVariable[Region, Mode, Cohort, "count"] = prism.export(initial_value = prism.Array[Region, Mode, Cohort, 'count'](0.0)) 
    inflow: prism.TimeVariable[Region, Mode, "count"]         = prism.export(initial_value = prism.Array[Region, Mode, 'count'](0.0))   
    outflow_by_cohort: prism.TimeVariable[Region, Mode, Cohort, "count"] = prism.export(initial_value = prism.Array[Region, Mode, Cohort, 'count'](0.0))

    def compute_initial_values(self, time: prism.Timeline):
        compute_historic(self.input_stock, self.survival_matrix, self.start_simulation,
                         self.stock_by_cohort, self.inflow, self.outflow_by_cohort)

    def compute_values(self, time: prism.Time):
        t, dt = time.t, time.dt
        stock_diff = self.input_stock.loc[t] - self.stock_by_cohort[t].sum("cohort")
        self.inflow[t] = stock_diff
        self.stock_by_cohort[t] = self.inflow[t]*self.survival_matrix[t, :]
        self.outflow_by_cohort[t] = self.stock_by_cohort[t]-self.stock_by_cohort[t-dt]

In [13]:
timeline = prism.Timeline(start=xr_vehicle_nr.coords["time"][0],
                          end=2060, stepsize=1)
timeline_simulate = prism.Timeline(start=1970,
                          end=2060, stepsize=1)

In [14]:
model = Stocks(timeline, start_simulation=1970, survival_matrix=survival_matrix, input_stock=xr_vehicle_nr)

In [15]:
model.simulate(timeline_simulate)